# Extracción de la API de Resultados Electorales (DINE)

Recolecta resultados de la **categoría Presidente** en **Santa Fe**.

Parte de *Análisis electoral Región Centro 2003-2023*. La salida respeta
`docs/CRITERIO_BASE_DE_DATOS.md`.

---

## Qué puede y qué no puede darte esta API

Esto se midió contra la API real antes de escribir el notebook. Leerlo
antes de correr nada, porque acota bastante las expectativas.

| Nivel | 2011 | 2015 | 2019 | 2023 |
|---|---|---|---|---|
| Provincia | agregado roto | sí | sí | sí |
| Departamento | parcial | 18 de 19 | **19 de 19** | 19 de 19 |
| Circuito | **no existe** | **no existe** | **no existe** | sí (523) |
| Mesa | no probado | no probado | no probado | no probado |

**El circuito solo existe para 2023.** Se verificó muestreando 999 códigos
sobre todo el espacio de ids: 2023 devuelve 384 circuitos, 2015 y 2019
devuelven cero. Se probó también pasando `seccionId` como padre, por si
hiciera falta: tampoco. No es un problema de descubrir códigos, es que el
dato no está publicado a ese nivel.

**Y para 2023 el circuito ya lo tenemos**, sacado del archivo por mesa. Así
que el aporte real de la API es el **nivel de departamento para 2011-2019**,
que hoy falta en la base.

## Dos límites que no cambian

1. **Es recuento PROVISORIO.** Difiere del escrutinio definitivo en torno al
   2 % en las fuerzas principales. No mezclar ambos en un mismo cálculo.
2. **No hay nada anterior a 2011.** 2003 y 2007 quedan afuera.

No requiere token: responde sin autenticación.

## 1. Configuración

In [ ]:
BASE = 'https://resultados.mininterior.gob.ar/api/resultados/getResultados'

DISTRITO_ID  = '21'   # Santa Fe
CATEGORIA_ID = '1'    # Presidente y Vice
TIPO_RECUENTO = '1'   # Provisional (el unico disponible)

INSTANCIAS = {'1': 'PASO', '2': 'GENERAL', '3': 'BALOTAJE'}
ANIOS = ['2011', '2015', '2019', '2023']

MAX_SECCION = 200     # las secciones de Santa Fe caen holgadamente aca
WORKERS = 16
TIMEOUT = 30
REINTENTOS = 3

## 2. Cliente

In [ ]:
import json, time, urllib.parse, urllib.request
from concurrent.futures import ThreadPoolExecutor


def consultar(anio, tipo_eleccion, seccion=None, circuito=None):
    params = {'anioEleccion': anio, 'tipoRecuento': TIPO_RECUENTO,
              'tipoEleccion': tipo_eleccion, 'categoriaId': CATEGORIA_ID,
              'distritoId': DISTRITO_ID}
    if seccion is not None:
        params['seccionId'] = str(seccion)
    if circuito is not None:
        params['circuitoId'] = str(circuito)

    url = f'{BASE}?{urllib.parse.urlencode(params)}'
    for intento in range(REINTENTOS):
        try:
            with urllib.request.urlopen(url, timeout=TIMEOUT) as r:
                return json.load(r)
        except Exception:
            if intento == REINTENTOS - 1:
                return None
            time.sleep(2 ** intento)


def tiene_datos(r):
    return bool(r and r.get('estadoRecuento', {}).get('mesasTotalizadas'))


def en_paralelo(fn, items, etiqueta=''):
    salida, total = [], len(items)
    with ThreadPoolExecutor(max_workers=WORKERS) as ex:
        for i, r in enumerate(ex.map(fn, items), 1):
            salida.append(r)
            if i % 50 == 0 or i == total:
                print(f'  {etiqueta} {i}/{total}', end='\r')
    print()
    return salida

## 3. Qué elecciones responden

In [ ]:
elecciones = []
for anio in ANIOS:
    for te, nombre in INSTANCIAS.items():
        if tiene_datos(consultar(anio, te)):
            elecciones.append((anio, te, nombre))
            print(f'{anio} {nombre}')
print(f'\n{len(elecciones)} elecciones con datos')

> El total que devuelve la consulta a nivel distrito **no es confiable**: en
> 2011 y a veces en 2023 entrega un subconjunto, o directamente cero. Sirve
> para saber qué existe, no como cifra de control. El control real es la
> suma de las partes, que se hace más abajo.

## 4. Recolección

Se barren las secciones (199 consultas por elección, cuestión de segundos) y
se recolecta de las que respondan. Para 2023 se agrega el nivel de circuito,
sembrando los 523 códigos ya relevados en el repositorio en vez de
redescubrirlos.

El barrido de circuitos para 2011-2019 **no se hace a propósito**: ese nivel
no existe en esos años y solo gastaría horas.

In [ ]:
import csv

RAW = ('https://raw.githubusercontent.com/abagilet12/'
       'An-lisis-electoral---Regi-n-Centro-2003---2023/'
       'claude/analisis-electoral-centro-tp8vwh/'
       'datos/procesados/nomenclador_circuitos_2023.csv')


def circuitos_conocidos(anio):
    if anio != '2023':
        return []
    try:
        with urllib.request.urlopen(RAW, timeout=30) as r:
            texto = r.read().decode('utf-8').splitlines()
        return [f['circuito_id'] for f in csv.DictReader(texto)]
    except Exception as e:
        print('no se pudieron leer los codigos conocidos:', e)
        return []


def filas_de(r, base):
    """Convierte una respuesta de la API en filas tidy."""
    e = r['estadoRecuento']
    base = {**base, 'mesas': e['mesasTotalizadas'],
            'electores': e['cantidadElectores'],
            'votantes': e['cantidadVotantes']}
    filas = []
    for a in r.get('valoresTotalizadosPositivos', []):
        filas.append({**base, 'agrupacion': a['nombreAgrupacion'],
                      'lista': '', 'tipo_registro': 'agrupacion',
                      'votos': a['votos']})
        for l in a.get('listas') or []:      # listas internas de las PASO
            filas.append({**base, 'agrupacion': a['nombreAgrupacion'],
                          'lista': l.get('nombre', ''),
                          'tipo_registro': 'lista', 'votos': l['votos']})
    otros = r.get('valoresTotalizadosOtros') or {}
    for campo, tipo in [('votosNulos', 'nulos'), ('votosEnBlanco', 'blancos'),
                        ('votosRecurridosComandoImpugnados', 'recurridos')]:
        if campo in otros:
            filas.append({**base, 'agrupacion': tipo, 'lista': '',
                          'tipo_registro': tipo, 'votos': otros[campo]})
    return filas


todas = []
for anio, te, nombre in elecciones:
    comun = {'anio': anio, 'instancia': nombre, 'cargo': 'PRESIDENTE Y VICE',
             'distrito_id': DISTRITO_ID, 'distrito': 'Santa Fe',
             'recuento_tipo': 'PROVISORIO', 'fuente': 'api_dine'}

    # --- departamento ---
    res = en_paralelo(lambda s: (s, consultar(anio, te, seccion=s)),
                      list(range(1, MAX_SECCION)), f'{anio} {nombre} secciones')
    for s, r in res:
        if tiene_datos(r):
            todas += filas_de(r, {**comun, 'nivel': 'seccion',
                                  'seccion_id': str(s), 'circuito_id': ''})

    # --- circuito (solo donde existe) ---
    circuitos = circuitos_conocidos(anio)
    if circuitos:
        res = en_paralelo(lambda c: (c, consultar(anio, te, circuito=c)),
                          circuitos, f'{anio} {nombre} circuitos')
        for c, r in res:
            if tiene_datos(r):
                todas += filas_de(r, {**comun, 'nivel': 'circuito',
                                      'seccion_id': '', 'circuito_id': c})
    print(f'{anio} {nombre}: {len(todas)} filas acumuladas\n')

## 5. Controles

**No exportar sin mirar esto.** El padrón se repite en cada fila de una misma
unidad, así que hay que deduplicar antes de sumar o queda multiplicado.

In [ ]:
import pandas as pd

df = pd.DataFrame(todas)

clave = ['anio', 'instancia', 'nivel', 'seccion_id', 'circuito_id']
unidades = df.drop_duplicates(clave)

resumen = []
for (anio, inst, nivel), g in df.groupby(['anio', 'instancia', 'nivel']):
    u = unidades[(unidades.anio == anio) & (unidades.instancia == inst)
                 & (unidades.nivel == nivel)]
    resumen.append({
        'anio': anio, 'instancia': inst, 'nivel': nivel,
        'unidades': len(u), 'mesas': u.mesas.sum(),
        'electores': u.electores.sum(),
        'positivos': g[g.tipo_registro == 'agrupacion'].votos.sum()})
print(pd.DataFrame(resumen).to_string(index=False))

**Cómo leerlo.** Santa Fe tiene **19 departamentos**. Si `unidades` da menos,
faltan datos: la API los omite, no es un error del barrido. Medido antes:
2019 da los 19 y cierra exacto contra el total provincial; 2015 da 18 y queda
134 mesas corto sobre 7.852; 2011 es claramente parcial.

Referencia de mesas en Santa Fe: 2015 → 7.852 · 2019 → 8.111 · 2023 → 8.332.

## 6. Exportar

In [ ]:
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    SALIDA = Path('/content/drive/MyDrive/analisis_electoral/api_dine')
except ImportError:
    SALIDA = Path('salida_api')

SALIDA.mkdir(parents=True, exist_ok=True)

for (anio, inst), g in df.groupby(['anio', 'instancia']):
    destino = SALIDA / f'{anio}_{inst}_santa_fe_presidente_api.csv'
    g.to_csv(destino, index=False, encoding='utf-8')
    print(f'-> {destino.name} ({len(g)} filas)')

df.to_csv(SALIDA / 'api_dine_santa_fe_consolidado.csv', index=False,
          encoding='utf-8')
print(f'\n-> consolidado: {len(df)} filas')

## 7. Después

Subir los CSV al repositorio en `datos/crudos/api_dine/` para integrarlos a
la serie.

**Al analizar, recordar:**

- Son **provisorios**; el nivel provincial de la base usa definitivo. Para eso
  están `recuento_tipo` y `fuente`: no mezclarlos en un mismo cálculo.
- Al sumar votos, filtrar `tipo_registro == 'agrupacion'`. Las listas internas
  de las PASO ya están dentro de su agrupación.
- Una elección recolectada en dos niveles aparece dos veces, una por nivel.
  Filtrar por `nivel` antes de agregar.
- Los códigos de circuito **no son comparables entre años**: Santa Fe los
  renumeró en 2023.
- Una serie temporal exige además **el mismo universo geográfico** todos los
  años. Ver `docs/HOMOLOGACION.md`.